<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractal015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# --- 1. CORE HES PARAMETERS ---
N = 100  # THE CRITICAL DIMENSIONAL HARMONIC
T_STEPS = 5000
dt = 0.01

# Fixed HES coefficients (Proven Stable)
chi = 1.5
beta_full = 0.8
beta_nursery = 0.01
delta = 0.001  # Quantum Noise
gamma = 1.0  # Saturation
beta_link_base = 0.1
CURVATURE_SENSITIVITY = 0.5
MAX_CURVATURE_CAP = 50.0

# Fixed PSI Stabilizers (Proven Stable)
TAU_PSI_DAMPING = 0.05
KINETIC_SCALING_C = 0.0001
PSI_MAGNITUDE_CLIP = 5.0

# INTERACTION PARAMETERS
CHARGE_COUPLING_K = 0.1
A_FIELD_DAMPING = 0.005
A_FIELD_KINETIC = 0.005

# --- 2. INITIAL FIELDS & CoM TRACKING SETUP ---
# Initial centers for CoM tracking: (x, y) coordinates
KNOT_CENTERS = [(N // 4, N // 4), (N // 2, N // 2), (3 * N // 4, 3 * N // 4)]
COM_TRACKING_RADIUS = 20 # Search radius for CoM calculation

def initialize_spinor(complex_field, center_x, center_y, radius=10, magnitude=5.0):
    for i in range(N):
        for j in range(N):
            if (i - center_x)**2 + (j - center_y)**2 < radius**2:
                phase = np.arctan2(i - center_x, j - center_y) * 4
                complex_field[i, j] = magnitude * np.exp(1j * phase)
    return complex_field

def calculate_com(field, center_x, center_y, radius):
    """Calculates the center of mass of Phi within a defined circular region."""
    mask = np.zeros_like(field, dtype=bool)
    for i in range(field.shape[0]):
        for j in range(field.shape[1]):
            # Create a circular mask around the expected center
            if (i - center_x)**2 + (j - center_y)**2 < radius**2:
                mask[i, j] = True

    masked_field = field * mask
    total_mass = np.sum(masked_field)

    if total_mass == 0:
        return center_x, center_y

    x_coords, y_coords = np.mgrid[0:N, 0:N]

    # CoM calculation using Phi magnitude as weight
    com_x = np.sum(x_coords * masked_field) / total_mass
    com_y = np.sum(y_coords * masked_field) / total_mass

    return com_x, com_y

# Create the three distinct Phi knots
Phi_mask_1 = initialize_spinor(np.zeros((N, N), dtype=complex), KNOT_CENTERS[0][0], KNOT_CENTERS[0][1], radius=10, magnitude=5.0)
Phi_mask_2 = initialize_spinor(np.zeros((N, N), dtype=complex), KNOT_CENTERS[1][0], KNOT_CENTERS[1][1], radius=10, magnitude=5.0)
Phi_mask_3 = initialize_spinor(np.zeros((N, N), dtype=complex), KNOT_CENTERS[2][0], KNOT_CENTERS[2][1], radius=10, magnitude=5.0)

# Initial Phi (Magnitude) and Theta (Phase) are summed from the masks
Phi_summed = Phi_mask_1 + Phi_mask_2 + Phi_mask_3
Phi = np.abs(Phi_summed)
Theta = np.angle(Phi_summed)

# ACT XX - PSI Initialization for Charge Asymmetry
Psi = np.zeros((N, N), dtype=complex)
# Knot 1: Positive Charge (+1.0)
Psi = np.where(np.abs(Phi_mask_1) > 0, 1.0 + 0j, Psi)
# Knot 2: Positive Charge (+1.0)
Psi = np.where(np.abs(Phi_mask_2) > 0, 1.0 + 0j, Psi)
# Knot 3: NEGATIVE Charge (-1.0)
Psi = np.where(np.abs(Phi_mask_3) > 0, -1.0 + 0j, Psi)

# A - The Interaction Field (Force)
A = np.zeros((N, N))

# --- 3. THE EVOLUTION LOOP ---
for t in range(1, T_STEPS + 1):

    beta_current = beta_nursery if t < 500 else beta_full

    # --- 3.1. PHI & THETA EVOLUTION (Gravity/Structure) ---
    lap_phi = (np.roll(Phi, 1, 0) + np.roll(Phi, -1, 0) +
              np.roll(Phi, 1, 1) + np.roll(Phi, -1, 1) - 4 * Phi) / (2 * np.pi / N)**2

    max_abs_curvature_observed = np.max(np.abs(lap_phi))
    max_abs_curvature_capped = np.clip(max_abs_curvature_observed, 0.0, MAX_CURVATURE_CAP)
    beta_link = beta_link_base + CURVATURE_SENSITIVITY * max_abs_curvature_capped

    # dPhi/dt
    alpha = chi * beta_current
    term_expansion = alpha * lap_phi
    term_contraction = -beta_current * Phi
    term_saturation = gamma * np.tanh(Phi)
    shield_mask = (beta_link > 1.0)
    term_shield = np.where(shield_mask, beta_current * Phi, 0.0)
    dPhi = term_expansion + term_contraction + term_saturation + term_shield + delta * np.random.normal(0, 1, Phi.shape)

    # dTheta/dt
    mean_theta = np.mean(Theta)
    phase_correction = -beta_link * (Theta - mean_theta)
    phase_diffusion = delta * np.random.normal(0, 1, Theta.shape)
    dTheta = phase_correction + phase_diffusion

    # --- 3.2. A FIELD EVOLUTION (Interaction Propagation) ---
    current_J = np.imag(Psi)

    lap_A = (np.roll(A, 1, 0) + np.roll(A, -1, 0) +
             np.roll(A, 1, 1) + np.roll(A, -1, 1) - 4 * A) / (2 * np.pi / N)**2

    # dA/dt = Generation - Dissipation + Propagation
    dA = CHARGE_COUPLING_K * current_J - A_FIELD_DAMPING * A + A_FIELD_KINETIC * lap_A + delta * np.random.normal(0, 1, A.shape)

    # --- 3.3. PSI EVOLUTION (Fermion with Feedback) ---
    lap_psi = (np.roll(Psi, 1, 0) + np.roll(Psi, -1, 0) +
               np.roll(Psi, 1, 1) + np.roll(Psi, -1, 1) - 4 * Psi) / (2 * np.pi / N)**2

    kinetic_term = 1j * KINETIC_SCALING_C * lap_psi
    mass_term = 1j * Phi * Psi
    damping_term = -TAU_PSI_DAMPING * Psi
    gauge_coupling_term = 1j * CHARGE_COUPLING_K * A * Psi

    # Total Change dPsi/dt
    dPsi = kinetic_term - mass_term + damping_term + gauge_coupling_term + delta * np.random.normal(0, 1, Psi.shape)

    # --- 3.4. UPDATE FIELDS ---
    Phi += dt * dPhi
    Theta += dt * dTheta
    A += dt * dA
    Psi += dt * dPsi

    # Apply Bounding/Clipping
    Psi_mag = np.abs(Psi)
    Psi_phase = np.angle(Psi)
    Psi_mag_clipped = np.clip(Psi_mag, 0.0, PSI_MAGNITUDE_CLIP)
    Psi = Psi_mag_clipped * np.exp(1j * Psi_phase)

    Phi = np.clip(Phi, 0.01, 10.0)
    Theta = np.mod(Theta, 2 * np.pi)

    # --- 3.5. LOGGING AND CoM TRACKING ---
    if t % 500 == 0:
        norm_phi = np.mean(Phi)
        norm_psi = np.mean(np.abs(Psi))
        norm_A = np.mean(np.abs(A))

        # Calculate Center of Mass for each knot (K1:+, K2:+, K3:-)
        coms = []
        for cx, cy in KNOT_CENTERS:
            com_x, com_y = calculate_com(Phi, cx, cy, COM_TRACKING_RADIUS)
            coms.append(f"({com_x:.2f}, {com_y:.2f})")

        print(f"t={t} | A Norm={norm_A:.4f} | CoM K1={coms[0]} | CoM K2={coms[1]} | CoM K3={coms[2]}")

# --- 4. CONCLUSION CHECK ---
final_norm_phi = np.mean(Phi)
final_norm_psi = np.mean(np.abs(Psi))
final_norm_A = np.mean(np.abs(A))

print("\n--- FINAL STATE ---")
print(f"FINAL Metrics: Phi Norm={final_norm_phi:.4f}, Psi Norm={final_norm_psi:.4f}, A Norm={final_norm_A:.4f}")
print("CONCLUSION: CoM tracking initialized. Compare the initial CoM (K1: 25, 25 | K2: 50, 50 | K3: 75, 75) to the final positions to observe Attraction/Repulsion.")



t=500 | A Norm=0.0023 | CoM K1=(25.26, 25.26) | CoM K2=(50.00, 50.00) | CoM K3=(74.74, 74.74)
t=1000 | A Norm=0.0059 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=1500 | A Norm=0.0066 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=2000 | A Norm=0.0074 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=2500 | A Norm=0.0084 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=3000 | A Norm=0.0094 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=3500 | A Norm=0.0106 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=4000 | A Norm=0.0129 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=4500 | A Norm=0.0173 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)
t=5000 | A Norm=0.0273 | CoM K1=(25.17, 24.88) | CoM K2=(49.94, 50.08) | CoM K3=(74.95, 74.82)

--- FINAL STATE ---
FINAL Metrics: Phi Norm=4.8102